In [ ]:
# If required, uncomment:
# %pip install pandas numpy scikit-learn matplotlib seaborn joblib

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

RANDOM_STATE = 42
TARGET = "Condition_Class"
CLASS_NAMES = ["Excellent", "Good", "Fair", "Poor"]

In [ ]:
TRAIN_PATH = "../training_image_features.csv"
VAL_PATH   = "../validation_image_features.csv"
TEST_PATH  = "../testing_image_features.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Testing:", test_df.shape)

for name, data in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"\\n{name} class distribution:")
    print(data[TARGET].value_counts())

In [ ]:
EXCLUDE_COLUMNS = [
    TARGET,
    "Sample_ID", "Image_ID",
    "Damage_Score", "Condition_Score",
    "Structural_Integrity", "Recovery_Pathway",
    "Reconditioning_Required", "Recovery_Potential",
    "Alternative_Use_Category", "Alternative_Use_Suitability",
    "pixel_img_width", "pixel_img_height"
]

feature_columns = [c for c in train_df.columns if c not in EXCLUDE_COLUMNS]

X_train = train_df[feature_columns].copy()
y_train = train_df[TARGET].astype(str)

X_val = val_df[feature_columns].copy()
y_val = val_df[TARGET].astype(str)

X_test = test_df[feature_columns].copy()
y_test = test_df[TARGET].astype(str)

numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [c for c in feature_columns if c not in numeric_features]

print("Features:", len(feature_columns))
print("Numeric:", len(numeric_features))
print("Categorical:", len(categorical_features))

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [ ]:
C_OPTIONS = [0.1, 1, 10, 100]
GAMMA_OPTIONS = ["scale", "auto"]

validation_results = []
candidate_models = {}

for C in C_OPTIONS:
    for gamma in GAMMA_OPTIONS:
        svm_pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("classifier", SVC(
                kernel="rbf",
                C=C,
                gamma=gamma,
                class_weight="balanced",
                probability=True,
                random_state=RANDOM_STATE
            ))
        ])

        svm_pipeline.fit(X_train, y_train)
        val_pred = svm_pipeline.predict(X_val)

        macro_f1 = f1_score(y_val, val_pred, average="macro", zero_division=0)
        accuracy = accuracy_score(y_val, val_pred)

        key = (C, gamma)
        candidate_models[key] = svm_pipeline
        validation_results.append({
            "C": C,
            "gamma": gamma,
            "validation_accuracy": accuracy,
            "validation_macro_f1": macro_f1
        })

results_df = pd.DataFrame(validation_results).sort_values(
    "validation_macro_f1", ascending=False
)
display(results_df)

best_row = results_df.iloc[0]
best_C = float(best_row["C"])
best_gamma = best_row["gamma"]
print("Selected C:", best_C)
print("Selected gamma:", best_gamma)

In [ ]:
X_train_val = pd.concat([X_train, X_val], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)

svm_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", SVC(
        kernel="rbf",
        C=best_C,
        gamma=best_gamma,
        class_weight="balanced",
        probability=True,
        random_state=RANDOM_STATE
    ))
])

svm_model.fit(X_train_val, y_train_val)
print("Final SVM model trained.")

In [ ]:
y_test_pred = svm_model.predict(X_test)

print("SVM Test Results")
print("----------------")
print("Accuracy:", round(accuracy_score(y_test, y_test_pred), 4))
print("Macro Precision:", round(precision_score(y_test, y_test_pred, average="macro", zero_division=0), 4))
print("Macro Recall:", round(recall_score(y_test, y_test_pred, average="macro", zero_division=0), 4))
print("Macro F1-score:", round(f1_score(y_test, y_test_pred, average="macro", zero_division=0), 4))

print("\\nClassification Report:")
print(classification_report(y_test, y_test_pred, labels=CLASS_NAMES, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_test_pred, labels=CLASS_NAMES)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("SVM – Test Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
os.makedirs("saved_models", exist_ok=True)

MODEL_PATH = os.path.join("saved_models", "codd_svm_pipeline.joblib")
joblib.dump(svm_model, MODEL_PATH)

print("Saved SVM pipeline to:", MODEL_PATH)
print("The saved pipeline includes preprocessing and the trained SVM.")